<a href="https://colab.research.google.com/github/BushDemon/ViT-prototype-classification/blob/main/experiments/ViT_mnist_4_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import base64
import io
import json
import math

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classes_to_keep = [4, 7]
image_size = 28
patch_size = 4
embedding_dim = 4
num_heads = 1
num_layers = 6
batch_size = 512
num_epochs = 120

print("Zariadenie:", device)

Zariadenie: cuda


In [2]:
# Embedding
class Embed(nn.Module):
    def __init__(self, h=28, w=28, p=4, d=3):
        super().__init__()
        self.embed = nn.Conv2d(1, d, kernel_size=(p,p), stride=(p,p), padding=0)
        self.position_encoding = nn.Parameter(torch.zeros(h//p, w//p, d).float())
    def forward(self, x):
        x = self.embed(x) # B x 1 x h x w -> B x d x h//p x w//p
        x = x.permute(0, 2, 3, 1) # -> B x h//p x w//p x d
        x += self.position_encoding.unsqueeze(0)
        return x

In [3]:
# Attention
class Attention(nn.Module):
    def __init__(self, d=3, num_heads=1):
        super(Attention, self).__init__()
        self.num_attention_heads = num_heads
        self.attention_head_size = d // self.num_attention_heads
        self.all_head_size = self.num_attention_heads * self.attention_head_size # == d

        self.query = nn.Linear(d, self.all_head_size)
        self.key = nn.Linear(d, self.all_head_size)
        self.value = nn.Linear(d, self.all_head_size)

        self.out = nn.Linear(self.all_head_size, d)
        #self.attn_dropout = nn.Dropout(0)
        #self.proj_dropout = nn.Dropout(0)

        self.softmax = nn.Softmax(dim=-1)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states):
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)

        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        attention_probs = self.softmax(attention_scores)
        weights = attention_probs
        #attention_probs = self.attn_dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)
        attention_output = self.out(context_layer)
        #attention_output = self.proj_dropout(attention_output)
        return attention_output, weights

In [4]:
def swish(x):
    return x * torch.sigmoid(x)

In [5]:
# perceptron
class Mlp(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc1 = nn.Linear(d, 4*d)
        self.fc2 = nn.Linear(4*d, d)
        self.act_fn = swish
    def forward(self, x):
        x = self.fc1(x)
        x = self.act_fn(x)
        x = self.fc2(x)
        return x

In [6]:
# transformation
class Block(nn.Module):
    def __init__(self, n=50, d=4, num_heads=1):
        super().__init__()
        self.attention_norm = nn.LayerNorm(d)
        self.attn = Attention(d, num_heads)
        self.ffn_norm = nn.LayerNorm(d)
        self.ffn = Mlp(d)
    def forward(self, x):
        h = x
        x = self.attention_norm(x)
        x, weights = self.attn(x)
        x += h
        h = x
        x = self.ffn_norm(x)
        x = self.ffn(x)
        x += h
        return x, weights

In [7]:
# wipeout
class Wipeout(nn.Module):
    def __init__(self, d=4, num_categories=10):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.wipeout = nn.Linear(d, num_categories, bias=False)
    def forward(self, x):
        return self.wipeout(self.norm(x)[:,0,:])

In [8]:
# Transformer (encoder-only)
class Transformer(nn.Module):
    def __init__(self, h=28, w=28, p=4, d=4, num_heads=1, num_layers=6, num_categories=10):
        super().__init__()
        self.embed = Embed(h,w,p,d)
        self.cls = nn.Parameter(torch.zeros(1,1,d).float())
        self.encoder = nn.ModuleList()
        for _ in range(num_layers):
            self.encoder.append(Block((h//p)*(w//p)+1, d, num_heads))
        self.wipeout = Wipeout(d,num_categories)
    def forward(self, x):
        hidden_states = self.embed(x) # -> B x (h/p) x (w/p) x d
        B, H, W, d = hidden_states.shape
        hidden_states = hidden_states.view(B,H*W,d) # -> B x (h/p)*(w/p) x d
        cls_tokens = self.cls.expand(B, -1, -1)  # B x 1 x d
        hidden_states = torch.cat([cls_tokens, hidden_states], dim=1) # -> B x (h/p)*(w/p)+1 x d
        attn_weights = []
        intermediates = [hidden_states]
        for block in self.encoder:
            hidden_states, weights = block(hidden_states)
            attn_weights.append(weights)
            intermediates.append(hidden_states)
        return self.wipeout(hidden_states), torch.stack(attn_weights).permute(1,0,2,3,4), torch.stack(intermediates).permute(1,0,2,3)
        # B x l, B x num_layers x num_heads x (h/p)*(w/p)+1 x (h/p)*(w/p)+1 x, B x num_layers+1 x (h/p)*(w/p)+1 x d

In [9]:
train_dataset = datasets.MNIST(
    root="data", train=True, download=True, transform=transforms.ToTensor()
)
test_dataset = datasets.MNIST(
    root="data", train=False, download=True, transform=transforms.ToTensor()
)

class_to_id = {digit: i for i, digit in enumerate(classes_to_keep)}
id_to_class = {i: digit for digit, i in class_to_id.items()}

100%|██████████| 9.91M/9.91M [00:00<00:00, 16.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 407kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.80MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 15.9MB/s]


In [10]:
class RemappedSubset(Dataset):
    def __init__(self, original_dataset, indices, class_to_new):
        self.original = original_dataset
        self.indices = indices
        self.class_to_new = class_to_new

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        x, y = self.original[real_idx]
        new_y = self.class_to_new[int(y)]
        return x, new_y

In [11]:
train_indices = [
    i for i, label in enumerate(train_dataset.targets.tolist())
    if label in class_to_id
]
test_indices = [
    i for i, label in enumerate(test_dataset.targets.tolist())
    if label in class_to_id
]
train_subset = RemappedSubset(train_dataset, train_indices, class_to_id)
test_subset = RemappedSubset(test_dataset, test_indices, class_to_id)

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
# Poradie musí zostať rovnaké pre prepojenie embeddingov s indexmi v MNIST.
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

print("Mapovanie tried:", class_to_id)
print(f"Tréning: {len(train_subset)} | Test: {len(test_subset)}")

Mapovanie tried: {4: 0, 7: 1}
Tréning: 12107 | Test: 2010


In [12]:
model = Transformer(
    h=image_size,
    w=image_size,
    p=patch_size,
    d=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    num_categories=len(classes_to_keep),
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)
scheduler = StepLR(optimizer, step_size=8, gamma=0.8)

def train(num_epochs):
    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits, _, _ = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * labels.size(0)
            train_correct += (logits.argmax(dim=1) == labels).sum().item()

        model.eval()
        test_correct = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                logits, _, _ = model(images)
                test_correct += (logits.argmax(dim=1) == labels).sum().item()

        print(
            f"Epocha {epoch:3}/{num_epochs} | "
            f"loss {train_loss / len(train_subset):.4f} | "
            f"train {train_correct / len(train_subset):.2%} | "
            f"test {test_correct / len(test_subset):.2%}"
        )
        scheduler.step()

In [13]:
print("Naučiteľné parametre:", sum(p.numel() for p in model.parameters()))
train(num_epochs)
torch.save(model.state_dict(), "mnist_vit4_47.pth")

Naučiteľné parametre: 1748
Epocha   1/120 | loss 0.7047 | train 50.08% | test 47.91%
Epocha   2/120 | loss 0.6922 | train 51.97% | test 51.14%
Epocha   3/120 | loss 0.6912 | train 52.33% | test 51.14%
Epocha   4/120 | loss 0.6868 | train 53.72% | test 51.24%
Epocha   5/120 | loss 0.6368 | train 73.59% | test 86.07%
Epocha   6/120 | loss 0.3479 | train 88.55% | test 90.35%
Epocha   7/120 | loss 0.2582 | train 92.23% | test 92.94%
Epocha   8/120 | loss 0.2151 | train 93.86% | test 94.43%
Epocha   9/120 | loss 0.1912 | train 94.46% | test 94.68%
Epocha  10/120 | loss 0.1763 | train 94.89% | test 95.17%
Epocha  11/120 | loss 0.1628 | train 95.23% | test 95.42%
Epocha  12/120 | loss 0.1487 | train 95.75% | test 95.77%
Epocha  13/120 | loss 0.1361 | train 96.05% | test 96.07%
Epocha  14/120 | loss 0.1313 | train 96.14% | test 95.87%
Epocha  15/120 | loss 0.1255 | train 96.27% | test 96.47%
Epocha  16/120 | loss 0.1161 | train 96.46% | test 96.52%
Epocha  17/120 | loss 0.1112 | train 96.54% |

In [ ]:
# Načítanie uloženého modelu. Pri použití tejto bunky preskoč tréningovú bunku vyššie.
# Najprv spusti bunky s definíciou architektúry a vytvorením modelu.
# Po vytvorení novej relácie Colabu nahraj uložený .pth súbor alebo nastav jeho cestu na Drive.
model_path = "mnist_vit4_47.pth"
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model.to(device)
model.eval()
print(f"Načítané váhy modelu z {model_path}. Model je pripravený na vyhodnotenie.")


In [14]:
# Tri ortonormálne smery v podpriestore x₁ + x₂ + x₃ + x₄ = 0.
sphere_basis = torch.tensor([
    [1, -1, 0, 0],
    [1, 1, -2, 0],
    [1, 1, 1, -3],
], dtype=torch.float32)
sphere_basis = F.normalize(sphere_basis, dim=1)

def sphere_coordinates(vectors, layer_norm):
    normalized = F.layer_norm(
        vectors.float().cpu(), (embedding_dim,), eps=layer_norm.eps
    )
    xyz = normalized @ sphere_basis.T
    lengths = xyz.norm(dim=1)
    valid = lengths > 1e-8
    if not valid.all():
        print(f"Vynechané vektory bez definovaného smeru: {(~valid).sum().item()}")
    if not valid.any():
        raise ValueError("Žiadny vektor nemá definovaný smer na guli.")
    return (xyz[valid] / lengths[valid, None]).numpy(), valid.numpy()

In [ ]:
def show_sphere(xyz, point_info, images, title, predictions=None, probabilities=None):
    # point_info: [index v images, index v MNIST, skutočná číslica, index patchu]
    # Predikcie aj pravdepodobnosti sú zoradené podľa obrázkov v images.
    # V patchovom grafe patrí predikcia celému obrázku.
    colors = {4: "#2563eb", 7: "#e07a20"}
    has_predictions = predictions is not None
    if has_predictions:
        predictions = np.asarray(predictions, dtype=int).tolist()
        if len(predictions) != len(images):
            raise ValueError("Ku každému obrázku musí patriť jedna predikcia.")
    if probabilities is not None:
        probabilities = np.asarray(probabilities, dtype=float).tolist()
        if not has_predictions or np.shape(probabilities) != (len(images), len(classes_to_keep)):
            raise ValueError("Pravdepodobnosti musia mať tvar počet obrázkov × počet tried.")
    if len(xyz) != len(point_info) or not point_info:
        raise ValueError("Každý bod musí mať svoj záznam v point_info.")

    fig = go.Figure()
    theta = np.linspace(0, 2 * np.pi, 101)
    circle = np.column_stack((np.cos(theta), np.sin(theta), np.zeros_like(theta)))
    for axes in [(0, 1, 2), (0, 2, 1), (2, 0, 1)]:
        ring = circle[:, axes]
        fig.add_trace(go.Scatter3d(
            x=ring[:, 0], y=ring[:, 1], z=ring[:, 2],
            mode="lines", line=dict(color="#b1bfcc", width=2),
            hoverinfo="skip", showlegend=False,
        ))

    point_trace_indices = []
    for digit in classes_to_keep:
        ids = [i for i, row in enumerate(point_info) if row[2] == digit]
        coordinates = xyz[ids]
        point_trace_indices.append(len(fig.data))
        fig.add_trace(go.Scatter3d(
            x=coordinates[:, 0], y=coordinates[:, 1], z=coordinates[:, 2],
            mode="markers", name=str(digit),
            marker=dict(size=3, color=colors[digit], opacity=0.85),
            customdata=[[i, point_info[i][1]] for i in ids],
            hovertemplate="Bod %{customdata[0]}<br>MNIST %{customdata[1]}<extra></extra>",
        ))

    axis = dict(range=[-1.08, 1.08], showspikes=False)
    fig.update_layout(
        title=title, height=520, template="plotly_white",
        margin=dict(l=0, r=0, b=0, t=50),
        legend_title_text="Skutočná číslica", uirevision="sphere",
        scene=dict(aspectmode="cube", xaxis=axis, yaxis=axis, zaxis=axis),
    )

    # PNG obrázky vznikajú v pamäti a vkladajú sa priamo do výstupu bunky.
    image_urls = []
    for image in images:
        buffer = io.BytesIO()
        pixels = image.squeeze(0).mul(255).round().clamp(0, 255).byte().numpy()
        Image.fromarray(pixels).save(buffer, format="PNG")
        image_urls.append("data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode("ascii"))

    preview_js = (
        "const points = " + json.dumps(point_info, separators=(",", ":")) + ";\n"
        "const images = " + json.dumps(image_urls, separators=(",", ":")) + ";\n"
        "const coordinates = " + json.dumps(np.asarray(xyz).tolist(), separators=(",", ":")) + ";\n"
        "const predictions = " + json.dumps(predictions, separators=(",", ":")) + ";\n"
        "const probabilities = " + json.dumps(probabilities, separators=(",", ":")) + ";\n"
        "const digits = " + json.dumps(classes_to_keep) + ";\n"
        "const pointTraces = " + json.dumps(point_trace_indices) + ";\n"
        "const grid = " + str(image_size // patch_size) + ";\n"
        "const patchSize = " + str(patch_size) + ";\n"
        + """
        const plot = document.getElementById('{plot_id}');
        const root = plot.closest('[data-sphere]');
        const panel = root.querySelector('[data-preview]');
        const img = panel.querySelector('img');
        const highlight = panel.querySelector('[data-patch]');
        const zoom = panel.querySelector('[data-zoom]');
        const canvas = panel.querySelector('canvas');
        const info = panel.querySelector('[data-info]');
        const input = panel.querySelector('input');
        const previewBody = panel.querySelector('[data-body]');
        const mode = root.querySelector('[data-color-mode]');
        const errorsOnly = root.querySelector('[data-errors-only]');
        const status = root.querySelector('[data-status]');
        const classColors = ['#2563eb', '#e07a20'];
        let visibleIds = [];
        let selectedPoint = null;

        function isError(index) {
            return predictions !== null && predictions[points[index][0]] !== points[index][2];
        }

        function drawPatch() {
            if (selectedPoint === null) return;
            const patch = points[selectedPoint][3];
            if (patch < 0 || !img.complete || img.naturalWidth === 0) return;
            const row = Math.floor(patch / grid);
            const col = patch % grid;
            const context = canvas.getContext('2d');
            context.imageSmoothingEnabled = false;
            context.clearRect(0, 0, patchSize, patchSize);
            context.drawImage(
                img, col * patchSize, row * patchSize, patchSize, patchSize,
                0, 0, patchSize, patchSize
            );
        }

        function selectPoint(index) {
            if (!Number.isInteger(index) || !visibleIds.includes(index)) {
                input.value = selectedPoint === null ? '' : selectedPoint;
                return;
            }
            selectedPoint = index;
            previewBody.hidden = false;
            const [imageId, mnistId, digit, patch] = points[index];
            input.value = index;
            const lines = ['MNIST index ' + mnistId, 'Skutočná číslica: ' + digit];
            if (predictions !== null) {
                const label = patch < 0 ? 'Predikcia' : 'Predikcia celého obrázka';
                lines.push(label + ': ' + predictions[imageId]);
                lines.push(isError(index) ? 'Výsledok: Nesprávne' : 'Výsledok: Správne');
            }
            if (probabilities !== null) {
                digits.forEach((value, i) => {
                    lines.push('P(' + value + ') = ' +
                        (100 * probabilities[imageId][i]).toFixed(2) + ' %');
                });
            }
            highlight.hidden = patch < 0;
            zoom.hidden = patch < 0;
            if (patch >= 0) {
                const row = Math.floor(patch / grid);
                const col = patch % grid;
                highlight.style.top = (100 * row / grid) + '%';
                highlight.style.left = (100 * col / grid) + '%';
                highlight.style.width = (100 / grid) + '%';
                highlight.style.height = (100 / grid) + '%';
                lines.push('Patch ' + (patch + 1) +
                    ' (riadok ' + (row + 1) + ', stĺpec ' + (col + 1) + ')');
            }
            info.textContent = lines.join('\\n');
            img.onload = drawPatch;
            img.src = images[imageId];
            drawPatch();
        }

        function refreshGraph() {
            visibleIds = points.map((_, i) => i).filter(i => !errorsOnly.checked || isError(i));
            const correctnessMode = mode.value === 'correctness';
            const groups = [[], []];
            visibleIds.forEach(i => {
                const imageId = points[i][0];
                const digit = mode.value === 'predicted' ? predictions[imageId] : points[i][2];
                const group = correctnessMode ? Number(isError(i)) : digits.indexOf(digit);
                groups[group].push(i);
            });
            const names = correctnessMode ? ['Správne', 'Nesprávne'] : digits.map(String);
            const colors = correctnessMode ? ['#94a3b8', '#dc2626'] : classColors;
            Plotly.update(plot, {
                x: groups.map(ids => ids.map(i => coordinates[i][0])),
                y: groups.map(ids => ids.map(i => coordinates[i][1])),
                z: groups.map(ids => ids.map(i => coordinates[i][2])),
                customdata: groups.map(ids => ids.map(i => [i, points[i][1]])),
                name: names,
                'marker.color': colors,
            }, {'legend.title.text': mode.options[mode.selectedIndex].text}, pointTraces);
            status.textContent = 'Zobrazené body: ' + visibleIds.length + ' / ' + points.length;
            if (visibleIds.length === 0) {
                selectedPoint = null;
                previewBody.hidden = true;
                info.textContent = 'Pre tento filter nie sú k dispozícii žiadne body.';
                input.value = '';
            } else {
                selectPoint(visibleIds.includes(selectedPoint) ? selectedPoint : visibleIds[0]);
            }
        }

        function selectPlotPoint(event) {
            const point = event.points[0];
            if (point && point.customdata) selectPoint(point.customdata[0]);
        }

        plot.on('plotly_hover', selectPlotPoint);
        plot.on('plotly_click', selectPlotPoint);
        input.addEventListener('change', () => selectPoint(input.valueAsNumber));
        mode.addEventListener('change', refreshGraph);
        errorsOnly.addEventListener('change', refreshGraph);
        refreshGraph();
        """
    )

    graph = fig.to_html(
        full_html=False, include_plotlyjs="cdn",
        config={"responsive": True, "displaylogo": False},
        post_script=preview_js,
    )
    disabled = "" if has_predictions else "disabled"
    html = f"""
    <div data-sphere style="font:14px sans-serif">
      <div style="display:flex;flex-wrap:wrap;align-items:center;gap:16px;padding:8px">
        <label>Farbenie
          <select data-color-mode>
            <option value="actual">Skutočná číslica</option>
            <option value="predicted" {disabled}>Predikovaná číslica</option>
            <option value="correctness" {disabled}>Správnosť</option>
          </select>
        </label>
        <label><input data-errors-only type="checkbox" {disabled}> Zobraziť iba chyby</label>
        <span data-status></span>
      </div>
      <div style="display:flex;flex-wrap:wrap;gap:16px">
        <aside data-preview style="padding:8px;width:230px;order:1">
          <p><strong>Vzorka vybraného bodu</strong></p>
          <p>Ukáž myšou na bod alebo naň klikni.</p>
          <div data-body>
            <div style="position:relative;width:196px;height:196px;background:#000">
              <img alt="Číslica MNIST"
                style="width:100%;height:100%;image-rendering:pixelated;display:block">
              <div data-patch hidden style="position:absolute;box-sizing:border-box;
                border:3px solid #ff5c35;pointer-events:none"></div>
            </div>
            <div data-zoom hidden>
              <p><strong>Zväčšený patch · {patch_size} × {patch_size} pixelov</strong></p>
              <canvas width="{patch_size}" height="{patch_size}" aria-label="Zväčšený vybraný patch"
                style="width:196px;height:196px;image-rendering:pixelated;background:#000"></canvas>
            </div>
          </div>
          <p data-info style="white-space:pre-line"></p>
          <label>ID bodu <input type="number" min="0" max="{len(point_info) - 1}"
            value="0" style="width:80px"></label>
        </aside>
        <div style="flex:1 1 470px;min-width:0">{graph}</div>
      </div>
    </div>
    """
    display(HTML(html))

In [ ]:
model.eval()
test_images = []
final_cls = []
test_logits = []
test_targets = []
with torch.no_grad():
    for images, labels in test_loader:
        logits, _, hidden_states = model(images.to(device))
        test_images.append(images)
        final_cls.append(hidden_states[:, -1, 0, :].cpu())
        test_logits.append(logits.cpu())
        test_targets.append(labels)

test_images = torch.cat(test_images)
final_cls = torch.cat(final_cls)
test_logits = torch.cat(test_logits)
test_targets = torch.cat(test_targets)
test_digits = [id_to_class[label] for label in test_targets.tolist()]
test_probabilities = test_logits.softmax(dim=1).numpy()
test_predictions = np.asarray([
    id_to_class[label] for label in test_logits.argmax(dim=1).tolist()
])
test_errors = test_predictions != np.asarray(test_digits)

correct_count = int((~test_errors).sum())
error_count = int(test_errors.sum())
print(f"Správne: {correct_count} | Chybné: {error_count} | Spolu: {len(test_digits)}")
print(f"Testovacia presnosť: {correct_count / len(test_digits):.2%}")
for actual, predicted in [(4, 7), (7, 4)]:
    count = int(((np.asarray(test_digits) == actual) & (test_predictions == predicted)).sum())
    print(f"Zámena {actual} → {predicted}: {count}")

xyz, valid = sphere_coordinates(final_cls, model.wipeout.norm)
point_info = [
    [i, test_indices[i], test_digits[i], -1]
    for i in np.flatnonzero(valid).tolist()
]
print(f"Vykreslené CLS embeddingy: {len(point_info)} / {len(test_images)}")
show_sphere(
    xyz, point_info, test_images, "Konečný CLS · MNIST 4 a 7",
    predictions=test_predictions, probabilities=test_probabilities,
)

In [ ]:
patch_block = 6
images_per_digit = 24
if not 1 <= patch_block <= num_layers:
    raise ValueError(f"Zvoľ patch_block od 1 do {num_layers}.")
if images_per_digit is not None and images_per_digit < 1:
    raise ValueError("images_per_digit musí byť kladné číslo alebo None.")

rng = np.random.default_rng(42)
digits = np.asarray(test_digits)
selected = []
for digit in classes_to_keep:
    candidates = np.flatnonzero(digits == digit)
    if images_per_digit is not None:
        candidates = rng.choice(
            candidates, size=min(images_per_digit, len(candidates)), replace=False
        )
    selected.extend(candidates.tolist())

patch_images = test_images[selected]
patch_tokens = []
model.eval()
with torch.no_grad():
    for images in patch_images.split(batch_size):
        _, _, hidden_states = model(images.to(device))
        # Vstup do bloku k je uložený na pozícii k - 1. Token 0 je CLS.
        patch_tokens.append(hidden_states[:, patch_block - 1, 1:, :].cpu())
patch_tokens = torch.cat(patch_tokens)
patches_per_image = patch_tokens.shape[1]

xyz, valid = sphere_coordinates(
    patch_tokens.flatten(0, 1), model.encoder[patch_block - 1].attention_norm
)
point_info = []
for index in np.flatnonzero(valid).tolist():
    image_slot, patch = divmod(index, patches_per_image)
    test_slot = selected[image_slot]
    point_info.append([image_slot, test_indices[test_slot], test_digits[test_slot], patch])

print(f"Obrázky: {len(patch_images)} | Patchov na obrázok: {patches_per_image}")
print(f"Vykreslené patche: {len(point_info)} / {len(patch_images) * patches_per_image}")
show_sphere(
    xyz, point_info, patch_images,
    f"Patch tokeny · normalizácia pred attention v bloku {patch_block}",
    predictions=test_predictions[selected],
    probabilities=test_probabilities[selected],
)